# Task 1 Notebook for Python Web Scraping Assignment 
### Caolán Maguire - 25256569

In [41]:
# Libraries import

# Library used for scraping pages and reading html
# !pip install beautifulsoup4
from bs4 import BeautifulSoup
import requests
import os
from pathlib import Path
import shutil
import json

#### explainer 

I create a list called 'quarter key' - I then set a variable that I later iterate for the quarter num and finally a variable for the page number, I run a while loop,where I request the page url with the variable changing for the quarter and for each quarter looping throught the pages available - if there' s a 404 not found returned moving on to the next quarter if it exists. If I get a positive response like a response 200 I then extract the information from that page using the BeautifulSoup library and save the data into JSON - reading from the page is quite easy as the structure remains consistent throughout.

URL Example: http://mlg.ucd.ie/modules/python/sources/rental/Q' + ' + quarter num + '-page' + ' + page number variable + '.html

In [45]:
# Variables for sources for web scraping
quarter_key = ['Q1', 'Q2', 'Q3', 'Q4']
quarter_num = 0
page_i = 1
consecutive_404s = 0
max_pages_per_quarter = 50  # Safety limit
total_requests = 0
max_total_requests = 200  # Global safety limit

# Store all listings organized by quarter
all_listings = {
    'Q1': [],
    'Q2': [],
    'Q3': [],
    'Q4': []
}

while True:
    # Safety check: total requests
    total_requests += 1
    if total_requests > max_total_requests:
        print("Safety limit reached: " + str(max_total_requests) + " requests made. Stopping.")
        break
    
    # Safety check: pages per quarter
    if page_i > max_pages_per_quarter:
        print("Too many pages for " + quarter + ". Moving to next quarter.")
        quarter_num += 1
        page_i = 1
        consecutive_404s += 1
        
        if consecutive_404s >= 4:
            print("All quarters exhausted. Stopping.")
            break
        continue
    
    page_num = str(page_i).zfill(2)
    quarter = quarter_key[quarter_num % 4]
    page = 'http://mlg.ucd.ie/modules/python/sources/rental/' + quarter + '-page' + str(page_num) + '.html'
    
    print("Request #" + str(total_requests) + ": " + page)
    
    try:
        response = requests.get(page, timeout=10)  # Add timeout
    except requests.exceptions.RequestException as e:
        print("Request failed: " + str(e))
        break
    
    if response.status_code == 404:
        print("Page not found (404)")
        consecutive_404s += 1
        
        # Move to next quarter, reset page number
        quarter_num += 1
        page_i = 1
        
        # If we've tried all 4 quarters and all returned 404, stop
        if consecutive_404s >= 4:
            print("No more pages found in any quarter. Stopping.")
            break
    
    elif response.status_code == 200:
        print("Extracting data...")
        consecutive_404s = 0  # Reset counter on success
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Find the ol element
        ol_element = soup.find('ol')
        if not ol_element:
            print("Warning: No <ol> found on page. Moving to next quarter.")
            quarter_num += 1
            page_i = 1
            continue
        
        # Find all <li> items
        listings = ol_element.find_all('li')
        
        if len(listings) == 0:
            print("No listings found. Moving to next quarter.")
            quarter_num += 1
            page_i = 1
            continue
        
        for listing in listings:
            # Extract the record type (date and property type)
            record_span = listing.find('span', class_='record')
            if not record_span:
                continue
                
            record = record_span.text.strip()
            
            # Extract all table rows
            rows = listing.find_all('tr')
            
            # Create dictionary for this listing
            listing_data = {
                'record': record,
                'page': page_num
            }
            
            # Extract data from each row
            for row in rows:
                cells = row.find_all('td')
                if len(cells) == 2:
                    key = cells[0].text.strip().replace(':', '')
                    value = cells[1].text.strip()
                    listing_data[key] = value
            
            # Add to the appropriate quarter
            all_listings[quarter].append(listing_data)
        
        print("Extracted " + str(len(listings)) + " listings (" + quarter + " total: " + str(len(all_listings[quarter])) + ")")
        
        # Move to next page in same quarter
        page_i += 1
    
    else:
        print("Unexpected status code: " + str(response.status_code) + ". Stopping.")
        break

# Display results
print("\n" + "="*60)
print("SCRAPING COMPLETE")
print("="*60)
print("Total requests made: " + str(total_requests))
for quarter in quarter_key:
    print(quarter + ": " + str(len(all_listings[quarter])) + " listings")

print("\nTotal listings: " + str(sum(len(all_listings[q]) for q in quarter_key)))

# Show example from Q1
if len(all_listings['Q1']) > 0:
    print("\nFirst Q1 listing:")
    for key, value in all_listings['Q1'][0].items():
        print("  " + key + ": " + str(value))

Request #1: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page01.html
Extracting data...
Extracted 20 listings (Q1 total: 20)
Request #2: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page02.html
Extracting data...
Extracted 20 listings (Q1 total: 40)
Request #3: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page03.html
Extracting data...
Extracted 20 listings (Q1 total: 60)
Request #4: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page04.html
Extracting data...
Extracted 20 listings (Q1 total: 80)
Request #5: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page05.html
Extracting data...
Extracted 20 listings (Q1 total: 100)
Request #6: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page06.html
Extracting data...
Extracted 20 listings (Q1 total: 120)
Request #7: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page07.html
Extracting data...
Extracted 20 listings (Q1 total: 140)
Request #8: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page08.html
Extra

Extracted 20 listings (Q3 total: 220)
Request #65: http://mlg.ucd.ie/modules/python/sources/rental/Q3-page12.html
Extracting data...
Extracted 20 listings (Q3 total: 240)
Request #66: http://mlg.ucd.ie/modules/python/sources/rental/Q3-page13.html
Extracting data...
Extracted 20 listings (Q3 total: 260)
Request #67: http://mlg.ucd.ie/modules/python/sources/rental/Q3-page14.html
Extracting data...
Extracted 20 listings (Q3 total: 280)
Request #68: http://mlg.ucd.ie/modules/python/sources/rental/Q3-page15.html
Extracting data...
Extracted 20 listings (Q3 total: 300)
Request #69: http://mlg.ucd.ie/modules/python/sources/rental/Q3-page16.html
Extracting data...
Extracted 20 listings (Q3 total: 320)
Request #70: http://mlg.ucd.ie/modules/python/sources/rental/Q3-page17.html
Extracting data...
Extracted 20 listings (Q3 total: 340)
Request #71: http://mlg.ucd.ie/modules/python/sources/rental/Q3-page18.html
Extracting data...
Extracted 20 listings (Q3 total: 360)
Request #72: http://mlg.ucd.ie/

Extracting data...
Extracted 4 listings (Q1 total: 1008)
Request #128: http://mlg.ucd.ie/modules/python/sources/rental/Q1-page27.html
Page not found (404)
Request #129: http://mlg.ucd.ie/modules/python/sources/rental/Q2-page01.html
Extracting data...
Extracted 20 listings (Q2 total: 515)
Request #130: http://mlg.ucd.ie/modules/python/sources/rental/Q2-page02.html
Extracting data...
Extracted 20 listings (Q2 total: 535)
Request #131: http://mlg.ucd.ie/modules/python/sources/rental/Q2-page03.html
Extracting data...
Extracted 20 listings (Q2 total: 555)
Request #132: http://mlg.ucd.ie/modules/python/sources/rental/Q2-page04.html
Extracting data...
Extracted 20 listings (Q2 total: 575)
Request #133: http://mlg.ucd.ie/modules/python/sources/rental/Q2-page05.html
Extracting data...
Extracted 20 listings (Q2 total: 595)
Request #134: http://mlg.ucd.ie/modules/python/sources/rental/Q2-page06.html
Extracting data...
Extracted 20 listings (Q2 total: 615)
Request #135: http://mlg.ucd.ie/modules/p

Extracting data...
Extracted 20 listings (Q4 total: 686)
Request #191: http://mlg.ucd.ie/modules/python/sources/rental/Q4-page13.html
Extracting data...
Extracted 20 listings (Q4 total: 706)
Request #192: http://mlg.ucd.ie/modules/python/sources/rental/Q4-page14.html
Extracting data...
Extracted 20 listings (Q4 total: 726)
Request #193: http://mlg.ucd.ie/modules/python/sources/rental/Q4-page15.html
Extracting data...
Extracted 20 listings (Q4 total: 746)
Request #194: http://mlg.ucd.ie/modules/python/sources/rental/Q4-page16.html
Extracting data...
Extracted 20 listings (Q4 total: 766)
Request #195: http://mlg.ucd.ie/modules/python/sources/rental/Q4-page17.html
Extracting data...
Extracted 20 listings (Q4 total: 786)
Request #196: http://mlg.ucd.ie/modules/python/sources/rental/Q4-page18.html
Extracting data...
Extracted 20 listings (Q4 total: 806)
Request #197: http://mlg.ucd.ie/modules/python/sources/rental/Q4-page19.html
Extracting data...
Extracted 20 listings (Q4 total: 826)
Reque

In [26]:

for i in ['Q1','Q2','Q3','Q4']:
    print(f'results from {i} are : {len(all_listings[i])}')

results from Q1 are : 1008
results from Q2 are : 990
results from Q3 are : 910
results from Q4 are : 886


#### Explainer

Above, I loop through the quarterly results and print lenth of the listing lists to ensure data has been entered, once this is verified, I move to below where I run through the directory - check my output directory and c

In [43]:
# SAVE DATA

# ensure directories exist as per output style
for directory in ['Q1','Q2','Q3','Q4']:
    dirpath = Path('output') / directory
    
    # Remove directory if it exists - to clean old data
    if dirpath.exists() and dirpath.is_dir():
        shutil.rmtree(dirpath)
    
    # Create the directory
    os.makedirs(dirpath)
    print(f"Created: {dirpath}")
    
    for index, item in enumerate(all_listings[directory]):
        print(f"Iteration {index}: {item}")
        with open(f'output/{directory}/data{index}.json', 'w') as f:
            json.dump(item, f)

Created: output\Q1
Iteration 0: {'record': 'January 2025 — Apartment', 'page': '01', 'Price': '840.00 per month', 'Location': 'Dublin City South — Dublin 12', 'Bedrooms': '1', 'Bathrooms': '2 Bathrooms', 'Parking': 'Y', 'Garden': '0', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 1: {'record': 'January 2025 — House', 'page': '01', 'Price': '€ 3,110.00', 'Location': 'Dublin City Nth. - D1', 'Bedrooms': '3 Bedrooms', 'Bathrooms': '1 Bathroom', 'Parking': 'True', 'Garden': 'Yes', 'Lease Length': '6 months', 'Contact': 'Owner'}
Iteration 2: {'record': 'January 2025 — Apartment', 'page': '01', 'Price': '€2,170 per month', 'Location': 'Dublin City South - Dublin 2', 'Bedrooms': '2', 'Bathrooms': '1 Bathroom', 'Parking': 'Not Available', 'Garden': 'N/A', 'Lease Length': '12 months', 'Contact': 'estate agent'}
Iteration 3: {'record': 'January 2025 — House', 'page': '01', 'Price': '€ 2,580', 'Location': 'Dublin City Sth. — D6', 'Bedrooms': '3 Bedrooms', 'Bathrooms': '2', 'Pa

Iteration 169: {'record': 'January 2025 — Apartment', 'page': '09', 'Price': '€2,690', 'Location': 'Dublin City South – Dublin 4', 'Bedrooms': '1', 'Bathrooms': '1', 'Parking': 'no', 'Garden': 'False', 'Lease Length': '3 months', 'Contact': 'estate  agent'}
Iteration 170: {'record': 'January 2025 —  Apartment', 'page': '09', 'Price': '€1,920', 'Location': 'Dublin City North – Dublin 5', 'Bedrooms': '1 Bedroom', 'Bathrooms': '2', 'Parking': 'N', 'Garden': '0', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 171: {'record': 'January 2025 — House', 'page': '09', 'Price': 'EUR 3,260 per month', 'Location': 'Dublin City South - Dublin 6W', 'Bedrooms': '4', 'Bathrooms': '1 Bathroom', 'Parking': 'N', 'Garden': 'Yes', 'Lease Length': '3 months', 'Contact': 'Owner'}
Iteration 172: {'record': 'January 2025 — Apartment', 'page': '09', 'Price': '1,350 per month', 'Location': 'Dublin City South - Dublin 12', 'Bedrooms': '1 Bedroom', 'Bathrooms': '2 Bathrooms', 'Parking': 'No', 'Ga

Iteration 335: {'record': 'February 2025 — Apartment', 'page': '17', 'Price': '€ 1,090', 'Location': 'Dublin City Sth. - D22', 'Bedrooms': '2 Bedrooms', 'Bathrooms': '2 Bathrooms', 'Parking': '', 'Garden': 'False', 'Lease Length': '6 months', 'Contact': 'Owner'}
Iteration 336: {'record': 'February 2025 — Apartment', 'page': '17', 'Price': 'EUR 1,830', 'Location': 'Dublin City South - Dublin 4', 'Bedrooms': '2 Bedrooms', 'Bathrooms': '1', 'Parking': '0', 'Garden': 'No', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 337: {'record': 'February 2025 — Apartment', 'page': '17', 'Price': '€ 1,830', 'Location': 'Dublin City North - Dublin 11', 'Bedrooms': '3 Bedrooms', 'Bathrooms': '1 Bathroom', 'Parking': 'N', 'Garden': 'N', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 338: {'record': 'February 2025 — House', 'page': '17', 'Price': '€ 3940 per month', 'Location': 'Dublin City North – Dublin 15', 'Bedrooms': '4 Bedrooms', 'Bathrooms': '4 Bathrooms', 'Pa

Iteration 504: {'record': 'January 2025 — Apartment', 'page': '01', 'Price': '840.00 per month', 'Location': 'Dublin City South — Dublin 12', 'Bedrooms': '1', 'Bathrooms': '2 Bathrooms', 'Parking': 'Y', 'Garden': '0', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 505: {'record': 'January 2025 — House', 'page': '01', 'Price': '€ 3,110.00', 'Location': 'Dublin City Nth. - D1', 'Bedrooms': '3 Bedrooms', 'Bathrooms': '1 Bathroom', 'Parking': 'True', 'Garden': 'Yes', 'Lease Length': '6 months', 'Contact': 'Owner'}
Iteration 506: {'record': 'January 2025 — Apartment', 'page': '01', 'Price': '€2,170 per month', 'Location': 'Dublin City South - Dublin 2', 'Bedrooms': '2', 'Bathrooms': '1 Bathroom', 'Parking': 'Not Available', 'Garden': 'N/A', 'Lease Length': '12 months', 'Contact': 'estate agent'}
Iteration 507: {'record': 'January 2025 — House', 'page': '01', 'Price': '€ 2,580', 'Location': 'Dublin City Sth. — D6', 'Bedrooms': '3 Bedrooms', 'Bathrooms': '2', 'Parking': '1'

Iteration 670: {'record': 'January 2025 — House', 'page': '09', 'Price': '4,080 per month', 'Location': 'Dublin City South - Dublin 16', 'Bedrooms': '4 Bedrooms', 'Bathrooms': '3 Bathrooms', 'Parking': '', 'Garden': '1', 'Lease Length': '12 months', 'Contact': 'Estate  Agent'}
Iteration 671: {'record': 'January 2025 — Apartment', 'page': '09', 'Price': '€ 3,510', 'Location': 'Dublin City South – Dublin 2', 'Bedrooms': '3', 'Bathrooms': '1 Bathroom', 'Parking': '', 'Garden': 'No', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 672: {'record': 'January 2025 —  House', 'page': '09', 'Price': 'EUR 3,880 per month', 'Location': 'Dublin City North – Dublin 5', 'Bedrooms': '3 Bedrooms', 'Bathrooms': '2 Bathrooms', 'Parking': 'True', 'Garden': 'Yes', 'Lease Length': '3 months', 'Contact': 'Estate Agent'}
Iteration 673: {'record': 'January 2025 — Apartment', 'page': '09', 'Price': '€2,690', 'Location': 'Dublin City South – Dublin 4', 'Bedrooms': '1', 'Bathrooms': '1', 'Parkin

...

Iteration 59: {'record': 'October 2025 — Apartment', 'page': '03', 'Price': '€ 2,520', 'Location': 'Dublin City South – Dublin 16', 'Bedrooms': '2 Bedrooms', 'Bathrooms': '1 Bathroom', 'Parking': 'no', 'Garden': 'False', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 60: {'record': 'October 2025 —  Apartment', 'page': '04', 'Price': '930', 'Location': 'Dublin City North - Dublin 1', 'Bedrooms': '1', 'Bathrooms': '2 Bathrooms', 'Parking': 'no', 'Garden': 'False', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 61: {'record': 'October 2025 — Apartment', 'page': '04', 'Price': '2,330 per month', 'Location': 'Dublin City South - Dublin 2', 'Bedrooms': '1', 'Bathrooms': '2 Bathrooms', 'Parking': '0', 'Garden': '0', 'Lease Length': '12.0', 'Contact': 'Owner'}
Iteration 62: {'record': 'October 2025 — House', 'page': '04', 'Price': 'EUR 3260', 'Location': 'Dublin City Nth. - D5', 'Bedrooms': '4', 'Bathrooms': '1 Bathroom', 'Parking': 'yes', 'Garden': '', 'L

Iteration 226: {'record': 'November 2025 — House', 'page': '12', 'Price': '€ 3,280 per month', 'Location': 'Dublin City Nth. — D3', 'Bedrooms': '1 Bedroom', 'Bathrooms': '2 Bathrooms', 'Parking': 'No', 'Garden': 'Yes', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 227: {'record': 'November 2025 — House', 'page': '12', 'Price': '4,150 per month', 'Location': 'Dublin City South – Dublin 8', 'Bedrooms': '1 Bedroom', 'Bathrooms': '2 Bathrooms', 'Parking': 'No', 'Garden': 'Yes', 'Lease Length': '6 months', 'Contact': 'Owner'}
Iteration 228: {'record': 'November 2025 — Apartment', 'page': '12', 'Price': '€ 1,990', 'Location': 'Dublin City North - Dublin 13', 'Bedrooms': '2 Bedrooms', 'Bathrooms': '2', 'Parking': '', 'Garden': 'N', 'Lease Length': '12.0', 'Contact': 'Estate Agent'}
Iteration 229: {'record': 'November 2025 — House', 'page': '12', 'Price': '€3,130 per month', 'Location': 'Dublin City South - Dublin 12', 'Bedrooms': '5 Bedrooms', 'Bathrooms': '3 Bathrooms', '

Iteration 401: {'record': 'December 2025 —  Apartment', 'page': '21', 'Price': '€1,360', 'Location': 'Dublin City South – Dublin 2', 'Bedrooms': '2 Bedrooms', 'Bathrooms': '2 Bathrooms', 'Parking': 'N', 'Garden': 'No', 'Lease Length': '12 months', 'Contact': 'Owner'}
Iteration 402: {'record': 'December 2025 — House', 'page': '21', 'Price': '3,220', 'Location': 'Dublin City South - Dublin 4', 'Bedrooms': '2 Bedrooms', 'Bathrooms': '2 Bathrooms', 'Parking': '', 'Garden': 'No', 'Lease Length': '12 months', 'Contact': 'Owner'}
Iteration 403: {'record': 'December 2025 — Apartment', 'page': '21', 'Price': '€ 1,600', 'Location': 'Dublin City Sth. – D2', 'Bedrooms': '1', 'Bathrooms': '1 Bathroom', 'Parking': '0', 'Garden': '—', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 404: {'record': 'December 2025 —  House', 'page': '21', 'Price': '€ 2,810 per month', 'Location': 'Dublin City South — Dublin 22', 'Bedrooms': '4 Bedrooms', 'Bathrooms': '2 Bathrooms', 'Parking': 'True', 

Iteration 578: {'record': 'October 2025 — Apartment', 'page': '07', 'Price': '€ 3080', 'Location': 'Dublin City Nth. - D1', 'Bedrooms': '2', 'Bathrooms': '1 Bathroom', 'Parking': '0', 'Garden': '—', 'Lease Length': '6 months', 'Contact': 'Estate Agent'}
Iteration 579: {'record': 'October 2025 — Apartment', 'page': '07', 'Price': '€ 1,920', 'Location': 'Dublin City South — Dublin 2', 'Bedrooms': '2', 'Bathrooms': '2', 'Parking': 'False', 'Garden': 'no', 'Lease Length': '12 months', 'Contact': 'Estate Agent'}
Iteration 580: {'record': 'October 2025 — Apartment', 'page': '07', 'Price': 'EUR 3,040', 'Location': 'Dublin City South – Dublin 2', 'Bedrooms': '2 Bedrooms', 'Bathrooms': '2', 'Parking': 'no', 'Garden': 'N', 'Lease Length': '12.0', 'Contact': 'Estate Agent'}
Iteration 581: {'record': 'October 2025 — House', 'page': '07', 'Price': '3,940 per month', 'Location': 'North Co Dublin', 'Bedrooms': '3 Bedrooms', 'Bathrooms': '2', 'Parking': 'N', 'Garden': '', 'Lease Length': '6 months', '

Iteration 755: {'record': 'November 2025 — Apartment', 'page': '16', 'Price': 'EUR 1840', 'Location': 'Dublin City Nth. - D9', 'Bedrooms': '1 Bedroom', 'Bathrooms': '1 Bathroom', 'Parking': 'No', 'Garden': 'No', 'Lease Length': '6 months', 'Contact': 'Estate Agent'}
Iteration 756: {'record': 'November 2025 — Apartment', 'page': '16', 'Price': 'EUR 1,240.00 per month', 'Location': 'Dublin City South - Dublin 10', 'Bedrooms': '2 Bedrooms', 'Bathrooms': '1', 'Parking': 'True', 'Garden': 'N', 'Lease Length': '12.0', 'Contact': 'Estate Agent'}
Iteration 757: {'record': 'November 2025 — Apartment', 'page': '16', 'Price': '€ 1500 per month', 'Location': 'Dublin City North - Dublin 1', 'Bedrooms': '2', 'Bathrooms': '2 Bathrooms', 'Parking': 'False', 'Garden': 'no', 'Lease Length': '12 months', 'Contact': 'Owner'}
Iteration 758: {'record': 'November 2025 — House', 'page': '16', 'Price': '€ 4,370.00', 'Location': 'Dublin City South - Dublin 22', 'Bedrooms': '3 Bedrooms', 'Bathrooms': '2 Bathroom

In [ ]:
# References

# https://oxylabs.io/blog/beautiful-soup-parsing-tutorial

# https://stackoverflow.com/questions/273192/how-do-i-create-a-directory-and-any-missing-parent-directories

# https://stackoverflow.com/questions/43765117/how-to-check-existence-of-a-folder-with-python-and-then-remove-it